In [ ]:
import ee
import pandas as pd

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-jaquelinevigolo')

In [ ]:
# Define the region of interest
asset_path = 'users/jaquelinevigolo/lm_bioma_250'
biomes = ee.FeatureCollection(asset_path)  # Load the asset as a FeatureCollection

# # Filter the feature collection by 'id' field for different regions
amazon = biomes.filter(ee.Filter.eq('CD_Bioma', 1))
caatinga = biomes.filter(ee.Filter.eq('CD_Bioma', 2))
cerrado = biomes.filter(ee.Filter.eq('CD_Bioma', 3))
atlanticForest = biomes.filter(ee.Filter.eq('CD_Bioma', 4))
pampa = biomes.filter(ee.Filter.eq('CD_Bioma', 5))
pantanal = biomes.filter(ee.Filter.eq('CD_Bioma', 6))


# asset_path = 'FAO/GAUL/2015/level0'
# brazil = ee.FeatureCollection(asset_path)  # Load the asset as a FeatureCollection

In [ ]:
lulc = ee.Image('projects/mapbiomas-public/assets/brazil/lulc/collection10/mapbiomas_brazil_collection10_coverage_v2')
# lulc.getInfo()

In [ ]:
forest_codes = [3, 4, 5, 6, 49]

In [ ]:
farming_codes = {
    "Pasture": 15,
    "Soybean": 39,
    "Sugarcane": 20,
    "Rice": 40,
    "Cotton": 62,
    "Other Temporary Crops": 41,
    "Coffee": 46,
    "Citrus": 47,
    "Palm Oil": 35,
    "Other Perennial Crops": 48,
    "Forest Plantation": 9,
    "Mosaic of Uses": 21,
    "Forest (2023)": 3,
    "Forest (2023)": 4,
    "Forest (2023)": 5,
    "Forest (2023)": 6,
    "Forest (2023)": 49
}


In [ ]:
biomes = {
    "Amazon": amazon.geometry(),
    "Caatinga": caatinga.geometry(),
    "Cerrado": cerrado.geometry(),
    "Atlantic Forest": atlanticForest.geometry(),
    "Pampa": pampa.geometry(),
    "Pantanal": pantanal.geometry()
}

In [ ]:
trend_image = ee.Image(
    "projects/ee-jaquelinevigolo/assets/trendPercent_CHIRPS_Rx5day_1987-2023")

In [ ]:
pos_trend = trend_image.gt(0)
neg_trend = trend_image.lt(0)

In [ ]:
lulc_1987 = lulc.select("classification_1987")
lulc_2023 = lulc.select("classification_2023")

In [ ]:
forest_1987 = lulc_1987.remap(
    forest_codes,
    [1] * len(forest_codes),
    0
)

In [ ]:
def forest_to_farming_area(code, biome_geom, trend_sign):

    mask = forest_1987.eq(1).And(lulc_2023.eq(code))

    area_img = ee.Image.pixelArea().updateMask(mask)

    trend_area_img = area_img.updateMask(trend_sign)

    stats = trend_area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=biome_geom,
        scale=30,
        bestEffort=True
    )

    return ee.Number(stats.get("area")).divide(1e6)  # km²


In [ ]:
# records = []

# for biome, biome_geom in biomes.items():

#     for trend_label, trend_sign in {
#         "Positive trend": pos_trend,
#         "Negative trend": neg_trend
#     }.items():

#         for class_name, code in farming_codes.items():

#             area_km2 = forest_to_farming_area(
#                 code,
#                 biome_geom,
#                 trend_sign
#             ).getInfo()

#             if area_km2 and area_km2 > 0:
#                 records.append({
#                     "biome": biome,
#                     "trend": trend_label,
#                     "source": "Forest (1987)",
#                     "target": class_name,
#                     "area_km2": area_km2
#                 })

# df = pd.DataFrame(records)

In [ ]:
# df.to_csv(r"D:\VSCode\MapBiomas-Award\dataSankeyDiagrama_R99pTOT_1987-2023.csv", index=False)

In [ ]:
df = pd.read_csv(r"D:\VSCode\MapBiomas-Award\dataSankeyDiagrama_R99pTOT_1987-2023.csv")

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=len(biomes),
    cols=2,
    specs=[[{"type": "sankey"}, {"type": "sankey"}]] * len(biomes),
    column_titles=["Positive rainfall trend", "Negative rainfall trend"],
    row_titles=list(biomes.keys())
)


In [ ]:
for i, biome in enumerate(biomes.keys(), start=1):

    for j, trend in enumerate(["Positive trend", "Negative trend"], start=1):

        sub = df[(df["biome"] == biome) & (df["trend"] == trend)]

        if sub.empty:
            continue

        labels = ["Forest (1987)"] + sub["target"].tolist()
        label_index = {l: k for k, l in enumerate(labels)}

        fig.add_trace(
            go.Sankey(
                node=dict(
                    pad=12,
                    thickness=12,
                    label=labels,
                    color=["#1f8d49"] + ["#ffcc80"] * len(sub)
                ),
                link=dict(
                    source=[label_index["Forest (1987)"]] * len(sub),
                    target=[label_index[t] for t in sub["target"]],
                    value=sub["area_km2"].tolist(),
                    color="rgba(120,120,120,0.45)"
                )
            ),
            row=i,
            col=j
        )


In [ ]:
fig.update_layout(
    title="R99pTOT",
    height=200 * len(biomes),
    width=600,
    font_size=12
)

fig.show()


In [ ]:
fig.write_image(r"D:\VSCode\MapBiomas-Award\Figs\sankeyDiagram_Rx5day_1987-2023.png", scale=2)